# RoPE

In [ ]:
import torch

token = torch.tensor([1.0, 2.0, 3.0, 4.0])

print(f"token = {token}")

d_k = 4

# print(torch.arange(0, 4, 2))

inv_freq = 1.0 / (10000**(torch.arange(0, d_k, 2).float()/d_k))

print(inv_freq)

token = tensor([1., 2., 3., 4.])
tensor([0, 2])
tensor([1.0000, 0.0100])


In [7]:
seq_len = 3

positions = torch.arange(seq_len).float()

print(positions)

angles = torch.outer(positions, inv_freq)

print(angles)

tensor([0., 1., 2.])
tensor([[0.0000, 0.0000],
        [1.0000, 0.0100],
        [2.0000, 0.0200]])


In [12]:
def rotate_half(x):
    print(x)

    x1 = x[..., ::2]
    print(x1)

    x2 = x[..., 1::2]
    print(x2)

    print(torch.stack((-x2, x1), dim=1))
    return torch.stack((-x2, x1), dim=1).flatten(-2)


In [13]:
print(rotate_half(token))

tensor([1., 2., 3., 4.])
tensor([1., 3.])
tensor([2., 4.])
tensor([[-2.,  1.],
        [-4.,  3.]])
tensor([-2.,  1., -4.,  3.])


In [21]:
def apply_rope(x):
    print(x.shape)

    seq_len = x.size(-2)
    d_k = x.size(-1)
    print(f"seq len = {seq_len} and d_k = {d_k}")

    inv_freq = 1.0/(10000 ** (torch.arange(0, d_k, 2, device=x.device).float()/d_k))
    print(f"inv freq = {inv_freq}")

    positions = torch.arange(seq_len, device=x.device).float()
    print(f"positions = {positions}")

    angles = torch.outer(positions, inv_freq)
    print(f"angles = {angles}")

    angles = torch.repeat_interleave(angles, 2, dim=-1)
    print(f"angles = {angles}")

    cos = torch.cos(angles)
    print(f"cos = {cos}")

    sin = torch.sin(angles)
    print(f"sin = {sin}")

    print(f"final output = {x + cos + rotate_half(x) * sin}")
    return (x * cos + rotate_half(x) * sin)


In [22]:
x = torch.tensor([
    [1., 2., 3., 4.],
    [1., 2., 3., 4.],
    [1., 2., 3., 4.]
])

apply_rope(x)

torch.Size([3, 4])
seq len = 3 and d_k = 4
inv freq = tensor([1.0000, 0.0100])
positions = tensor([0., 1., 2.])
angles = tensor([[0.0000, 0.0000],
        [1.0000, 0.0100],
        [2.0000, 0.0200]])
angles = tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [1.0000, 1.0000, 0.0100, 0.0100],
        [2.0000, 2.0000, 0.0200, 0.0200]])
cos = tensor([[ 1.0000,  1.0000,  1.0000,  1.0000],
        [ 0.5403,  0.5403,  0.9999,  0.9999],
        [-0.4161, -0.4161,  0.9998,  0.9998]])
sin = tensor([[0.0000, 0.0000, 0.0000, 0.0000],
        [0.8415, 0.8415, 0.0100, 0.0100],
        [0.9093, 0.9093, 0.0200, 0.0200]])
tensor([[1., 2., 3., 4.],
        [1., 2., 3., 4.],
        [1., 2., 3., 4.]])
tensor([[1., 3.],
        [1., 3.],
        [1., 3.]])
tensor([[2., 4.],
        [2., 4.],
        [2., 4.]])
tensor([[[-2., -4.],
         [ 1.,  3.]],

        [[-2., -4.],
         [ 1.,  3.]],

        [[-2., -4.],
         [ 1.,  3.]]])
final output = tensor([[ 2.0000,  3.0000,  4.0000,  5.0000],
    

tensor([[ 1.0000,  2.0000,  3.0000,  4.0000],
        [-1.1426, -2.2853,  3.0098,  4.0298],
        [-2.2347, -4.4695,  3.0194,  4.0592]])

In [25]:
import torch

tensor_a = torch.ones(1, 2, 1, 4)  # Shape: (1, 2, 1, 4)
tensor_b = torch.zeros(1, 2, 3, 4) # Shape: (1, 2, 3, 4)
print(tensor_a)
print(tensor_b)

# Concatenate along dim=-2
output = torch.cat((tensor_a, tensor_b), dim=-2)

print("Output Shape:", output.shape)
print(output)
# Output: torch.Size([1, 2, 4, 4])


tensor([[[[1., 1., 1., 1.]],

         [[1., 1., 1., 1.]]]])
tensor([[[[0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]],

         [[0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]]]])
Output Shape: torch.Size([1, 2, 4, 4])
tensor([[[[1., 1., 1., 1.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]],

         [[1., 1., 1., 1.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.],
          [0., 0., 0., 0.]]]])
